# ML-02 — Research Question and Provisional Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/YOUR-USERNAME/flyrank-ml-internship/blob/main/work/notebooks/w01_research_question.ipynb)

**Provisional lane: Lane 3 — Structured Content Archetype Clustering.**

> Skills for this card: `skills/framing-ml-problems/SKILL.md` + `skills/flyrank/flyrank-data/SKILL.md`
> (router: `skills/README.md`). Data: `data/raw/content_refresh_anonymized.csv` — 30,000 rows × 44 columns,
> 32 pseudonymized clients, trailing-90-day window. Column reference: `docs/data-dictionary.md`.

Every number below is computed live from the shipped CSV in the cell above it. Nothing is typed in by hand.
Where I made a choice — a cutoff, a feature, a value of k — it is labelled as a choice, not a fact.

## 1. My lane (or freestyle) and why

**Lane 3 — Structured Content Archetype Clustering.**

> What performance archetypes exist across the content inventory, and what treatment does each one
> deserve?

Three reasons I picked it.

**The inventory is too big to treat page by page, and too varied to treat as one thing.** A content team
does not make 18,000 individual decisions. They make a handful of *policies* — protect this kind of page,
rewrite that kind, merge these, leave those alone — and then apply them. Clustering matches that shape of
work exactly: it produces a small number of groups a human can hold in their head, each with an action
attached.

**The alternative lanes assume I already know what I'm looking for.** Lanes 2 and 4 score pages against a
target I define up front — "declining", "under-capturing". That is a good approach when you know which
failure you care about. I don't yet. Clustering asks the prior question: *what kinds of pages are there
at all?* If the answer turns out to be "four kinds, and two of them are being treated identically by the
current rules", that is worth more than a better-tuned score for a failure mode I guessed at.

**There is no label, so there is no leakage minefield.** Every supervised lane in this internship lives or
dies on window discipline — feature windows that must not touch target windows, 90-day means that quietly
contain the answer. Unsupervised work has none of that. What it has instead is a harder honesty problem,
which section 4 is about: clusters will always come out of the algorithm, and it is entirely on me to
show they mean something.

**Grain (what one row means):** one pseudonymized content item (a page), belonging to one pseudonymized
client, summarised over a trailing 90-day window. `content_id` and `client_id` are pseudonyms — used for
grouping and joins, never as features.

In [1]:
# --- Setup: works in Colab and locally, from repo root or from work/notebooks/ ---
import os, sys, subprocess
from pathlib import Path
import numpy as np, pandas as pd, matplotlib.pyplot as plt

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/huzaifaguru/flyrank-ml-internship"   # <-- change to YOUR fork
CSV_REL  = "data/raw/content_refresh_anonymized.csv"

if IN_COLAB and not Path(CSV_REL).exists():
    if not os.path.isdir("flyrank-ml-internship"):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, "flyrank-ml-internship"], check=True)
    os.chdir("flyrank-ml-internship")
else:
    here = Path.cwd()
    for c in [here, *here.parents]:
        if (c / CSV_REL).exists():
            os.chdir(c); break
assert Path(CSV_REL).exists(), f"starter CSV not found from {Path.cwd()}"
pd.set_option("display.width", 180); pd.set_option("display.max_columns", 40)

df = pd.read_csv(CSV_REL)
print(f"rows x columns       : {df.shape[0]:,} x {df.shape[1]}")
print(f"pseudonymized clients: {df['client_id'].nunique()}")
print(f"unique pages         : {df['content_id'].nunique():,}")
print(f"duplicate content_id : {df['content_id'].duplicated().sum()}  <- must be 0 for my grain to hold")

rows x columns       : 30,000 x 44
pseudonymized clients: 32
unique pages         : 30,000
duplicate content_id : 0  <- must be 0 for my grain to hold


## 2. The question: decision, action, cost of a wrong call

### The decision
Not "is this page good?" and not "which page first?" — those are other lanes. The decision here is
**"what kind of page is this, and therefore what treatment does it get?"** It is a routing decision, made
once per archetype and applied to thousands of pages, rather than a judgement made one page at a time.

### The output
**Cluster profiles plus an action mapping.** For each archetype: how many pages, what its typical numbers
look like in plain units, a name that came *after* someone read the profile, one recommended action, and
an explicit note on what would make that action wrong.

### Who acts, and what they do
A content lead sets policy. The action vocabulary is the lane guide's:

| Action | When |
|---|---|
| **protect** | the archetype is performing — don't touch it, monitor for decline |
| **improve** | real exposure, weak conversion — worth editorial attention |
| **rewrite** | the content shape looks wrong for the demand it attracts |
| **merge** | overlapping pages splitting the same demand |
| **prune** | sustained cost, no return |
| **monitor** | not enough evidence to act — the honest default |

### The cost of a wrong recommendation
This is where clustering is *more* dangerous than ranking, and I want that on the record in week 2.

A ranked queue is wrong one page at a time — a reviewer opens it, sees nothing wrong, closes it, and has
lost an hour. **An archetype is wrong thousands of pages at a time.** If I label a 4,000-page cluster
"prune" and the cluster is really "seasonal but healthy", the recommendation destroys traffic that was
working, at scale, before anyone notices. The asymmetry is severe: a wrong "protect" costs a missed
opportunity; a wrong "prune" costs the asset itself.

Two consequences I am building in from the start: **`prune` requires evidence beyond cluster membership**,
and **`monitor` is a legitimate action** for any archetype I cannot characterise confidently.

### Why data or ML helps at all
An honest answer has to admit the alternative works reasonably well. A human *can* write a grouping rule —
"over 3,000 impressions and on page one is a champion; not updated in 90 days and still visible is stale" —
and section 3 shows that rule already sorts the inventory into sensible piles. So the case for clustering
is not that rules fail. It is narrower:

> A hand-written ladder puts a page in the first bucket whose threshold it crosses. It cannot notice that
> **one of its buckets contains two different kinds of page**, because it never looks at the shape of what
> lands inside. Clustering can, because it has no buckets to begin with.

Section 3 measures whether that actually happens on this data. It does.

## 3. Quick look at the data (2-3 real numbers)

Three numbers, in the order that makes the argument.

### Number 1 — the inventory is big enough, and I have to choose who is in it

In [2]:
# Policy choice: a page needs enough exposure for its 90-day behaviour to mean something.
# 300 impressions is not arbitrary - it is the documented "moderate" impression_tier boundary
# in docs/data-dictionary.md, so I am borrowing a threshold the dataset already defines.
MIN_IMPRESSIONS = 300

for thr in (100, 300, 500, 1000):
    sub = df[(df["impressions_90d"] >= thr) & (df["avg_position"] > 0)]
    mark = "  <-- my choice" if thr == MIN_IMPRESSIONS else ""
    print(f"  impressions_90d >= {thr:5d} & has position data : {len(sub):6,} pages, "
          f"{sub['client_id'].nunique()} clients{mark}")

elig = df[(df["impressions_90d"] >= MIN_IMPRESSIONS) & (df["avg_position"] > 0)].copy()
print()
print(f"avg_position == 0 rows excluded (means NO DATA, not rank zero): "
      f"{int((df['avg_position'] == 0).sum()):,}")
print(f"Eligible inventory: {len(elig):,} pages ({len(elig)/len(df)*100:.1f}% of the CSV) "
      f"across {elig['client_id'].nunique()} clients")

  impressions_90d >=   100 & has position data : 22,006 pages, 30 clients
  impressions_90d >=   300 & has position data : 18,752 pages, 29 clients  <-- my choice
  impressions_90d >=   500 & has position data : 16,726 pages, 28 clients
  impressions_90d >=  1000 & has position data : 13,512 pages, 28 clients

avg_position == 0 rows excluded (means NO DATA, not rank zero): 1,205
Eligible inventory: 18,752 pages (62.5% of the CSV) across 29 clients


**Read:** 18,752 pages across 29 clients — 62.5% of the slice. Big enough that per-page decisions are
not realistic, which is the premise of the whole lane, and spread across enough clients that whatever I
find is not one website's habits.

### Number 2 — a hand-written rule already sorts this inventory, and I can show where it stops seeing

In [3]:
# The obvious rule a human would write. First match wins - readable on purpose.
def rule_archetype(r):
    if r["impressions_90d"] >= 3000 and r["avg_position"] <= 10:      return "CHAMPION"
    if r["days_since_last_update"] >= 90 and r["impressions_90d"] >= 500: return "STALE_VISIBLE"
    if r["engagement_rate"] >= 10:                                     return "ENGAGED_NICHE"
    if r["days_with_impressions"] < 45:                                return "INTERMITTENT"
    return "STEADY_LOW"

elig["rule_archetype"] = elig.apply(rule_archetype, axis=1)
print("What the hand-written ladder produces:\n")
print(elig.groupby("rule_archetype").agg(
    pages=("content_id", "size"),
    impressions=("impressions_90d", "median"), clicks=("clicks_90d", "median"),
    ctr=("ctr", "median"), position=("avg_position", "median"),
    age_days=("content_age_days", "median"), since_update=("days_since_last_update", "median"),
).sort_values("pages", ascending=False).round(2).to_string())
print()
big = elig["rule_archetype"].value_counts().idxmax()
n_big = int(elig["rule_archetype"].value_counts().max())
print(f"'{big}' is {n_big:,} pages - {n_big/len(elig)*100:.0f}% of the inventory in ONE bucket,")
print("defined entirely by what it is NOT. That is the bucket worth looking inside.")

What the hand-written ladder produces:

                pages  impressions  clicks   ctr  position  age_days  since_update
rule_archetype                                                                    
STEADY_LOW       8130       1194.0     2.0  0.13      15.1     223.0          20.0
STALE_VISIBLE    4891       2200.0     3.0  0.13      18.5     238.0         104.0
CHAMPION         4571       9614.0    27.0  0.26       6.0     236.0          22.0
ENGAGED_NICHE     903       1314.0     3.0  0.21      14.0     228.0          20.0
INTERMITTENT      257        517.0     1.0  0.03      12.2     175.0          20.0

'STEADY_LOW' is 8,130 pages - 43% of the inventory in ONE bucket,
defined entirely by what it is NOT. That is the bucket worth looking inside.


In [4]:
# Split that bucket by content age alone and see whether it holds together.
steady = elig[elig["rule_archetype"] == "STEADY_LOW"]
young = steady[steady["content_age_days"] < 200]
old = steady[steady["content_age_days"] >= 200]
print(f"Inside the '{big}' bucket, split at 200 days old:\n")
print(f"  younger than 200 days : {len(young):,} pages | median age "
      f"{young['content_age_days'].median():.0f}d | median CTR {young['ctr'].median():.2f}% | "
      f"median position {young['avg_position'].median():.1f}")
print(f"  200 days or older     : {len(old):,} pages | median age "
      f"{old['content_age_days'].median():.0f}d | median CTR {old['ctr'].median():.2f}% | "
      f"median position {old['avg_position'].median():.1f}")
print()
print("Same bucket, same recommended treatment under the rule - but a page that is four months")
print("old and still finding its feet is not the same asset as one that has had two years and")
print("settled here. One might be early; the other has had its chance. The rule cannot tell them")
print("apart because age is not in the rule, and adding age would just create a new fixed edge.")

Inside the 'STEADY_LOW' bucket, split at 200 days old:

  younger than 200 days : 4,000 pages | median age 119d | median CTR 0.17% | median position 13.4
  200 days or older     : 4,130 pages | median age 441d | median CTR 0.10% | median position 16.9

Same bucket, same recommended treatment under the rule - but a page that is four months
old and still finding its feet is not the same asset as one that has had two years and
settled here. One might be early; the other has had its chance. The rule cannot tell them
apart because age is not in the rule, and adding age would just create a new fixed edge.


**Read:** the ladder's largest bucket holds **8,130 pages — 43% of the inventory** — and it is defined
by exclusion: everything that isn't a champion, isn't stale, isn't engaged, isn't intermittent. Split it on
age alone and the halves have visibly different profiles. That is the specific gap clustering fills: not
"beat the rule's accuracy", but *see structure inside a bucket the rule treats as uniform*.

### Number 3 — the structure is real and reproducible, not an artefact of the algorithm

K-Means will return clusters from pure noise. The only question worth asking in week 2 is whether these
survive being re-run.

In [5]:
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score, adjusted_rand_score

elig["log_impressions"] = np.log1p(elig["impressions_90d"])   # traffic is heavy-tailed
elig["log_clicks"] = np.log1p(elig["clicks_90d"])
elig["log_sessions"] = np.log1p(elig["sessions_90d"])
elig["impression_consistency"] = elig["days_with_impressions"] / 90

FEATURES = ["log_impressions", "log_clicks", "ctr", "avg_position", "log_sessions",
            "engagement_rate", "impression_consistency", "content_age_days",
            "days_since_last_update"]
X = StandardScaler().fit_transform(elig[FEATURES].fillna(elig[FEATURES].median()))

sample = np.random.RandomState(0).choice(len(X), 5000, replace=False)
print("k selection - silhouette (higher = better separated) on a 5,000-page sample:\n")
for k in range(3, 9):
    km = KMeans(n_clusters=k, n_init=25, random_state=42).fit(X)
    sizes = sorted(pd.Series(km.labels_).value_counts().tolist())
    print(f"  k={k}  silhouette={silhouette_score(X[sample], km.labels_[sample]):.3f}  "
          f"smallest cluster={sizes[0]:5,}  sizes={sizes}")

K = 6   # policy choice: the silhouette peak, and the smallest cluster is still 537 pages
base = KMeans(n_clusters=K, n_init=25, random_state=42).fit_predict(X)
aris = [adjusted_rand_score(base, KMeans(n_clusters=K, n_init=25, random_state=s).fit_predict(X))
        for s in (1, 7, 13, 99, 2024)]
print(f"\nSTABILITY at k={K} - agreement (ARI) between the k=42 run and 5 other random seeds:")
print(f"  {np.round(aris, 3).tolist()}")
print(f"  mean {np.mean(aris):.3f}  (1.0 = identical partition, 0.0 = no better than chance)")

k selection - silhouette (higher = better separated) on a 5,000-page sample:

  k=3  silhouette=0.186  smallest cluster=4,706  sizes=[4706, 6797, 7249]
  k=4  silhouette=0.209  smallest cluster=3,837  sizes=[3837, 4621, 4715, 5579]
  k=5  silhouette=0.215  smallest cluster=1,012  sizes=[1012, 3767, 4290, 4725, 4958]
  k=6  silhouette=0.221  smallest cluster=  537  sizes=[537, 985, 3595, 4113, 4598, 4924]
  k=7  silhouette=0.218  smallest cluster=  524  sizes=[524, 918, 1164, 3460, 3904, 4262, 4520]
  k=8  silhouette=0.220  smallest cluster=  491  sizes=[491, 868, 1060, 1509, 3052, 3833, 3945, 3994]

STABILITY at k=6 - agreement (ARI) between the k=42 run and 5 other random seeds:
  [0.999, 1.0, 1.0, 0.995, 0.997]
  mean 0.998  (1.0 = identical partition, 0.0 = no better than chance)


**Read — and this is the number that justifies seven weeks.** Silhouette peaks at **k = 6 (0.221)**, and
re-running from five different random seeds reproduces essentially the same partition every time
(**ARI 0.995–1.000**). The groups are not an artefact of where the algorithm happened to start.

**The honest half of that.** A silhouette of 0.221 is **weak separation**. These are not crisp, well-spaced
clusters — they are regions of a continuum with soft edges. That is normal for behavioural metrics and it
is exactly why the lane guide says to treat clusters as *a lens, not true labels*. I am writing that down
now, in week 2, so that nothing later in this project quietly upgrades "a useful way to group pages" into
"the natural kinds of content."

### One thing I found while looking, which will constrain the whole project

In [6]:
print("engagement_rate == 0 across the eligible inventory: "
      f"{(elig['engagement_rate'] == 0).mean()*100:.0f}% of pages "
      f"({int((elig['engagement_rate'] == 0).sum()):,})")
print(f"...but sessions_90d == 0 for only {(elig['sessions_90d'] == 0).mean()*100:.0f}% of them.\n")
print("So 'zero engagement' is not 'nobody engaged'. Look at how it moves with volume:\n")
print(elig.groupby("impression_tier").agg(
    pages=("content_id", "size"),
    pct_with_any_engagement=("engagement_rate", lambda s: (s > 0).mean() * 100),
    median_engagement_when_present=("engagement_rate", lambda s: s[s > 0].median()),
).round(1).to_string())

engagement_rate == 0 across the eligible inventory: 59% of pages (11,060)
...but sessions_90d == 0 for only 0% of them.

So 'zero engagement' is not 'nobody engaged'. Look at how it moves with volume:

                 pages  pct_with_any_engagement  median_engagement_when_present
impression_tier                                                                
excellent         1078                     89.6                             2.5
good              7205                     62.2                             3.8
moderate         10469                     21.4                             7.7


**Read:** 59% of eligible pages show `engagement_rate = 0`, and the share with *any* engagement climbs
**21% → 62% → 90%** as impression volume rises. A page with six sessions is very likely to record zero
engaged sessions by arithmetic, not by behaviour.

That matters for this lane specifically: if I feed `engagement_rate` into the clustering as-is, it will
partly encode **volume** under a different name, and I will get a cluster that I am tempted to call
"disengaged content" when it is really "small content". I am keeping the feature — it does isolate a real
minority — but ML-03 has to state what that cluster can and cannot be called.

## 4. Careful words: what I can and can't claim

### What this work will be able to say
- **Observed.** "Across 18,752 pages from 29 pseudonymized clients, K-Means at k=6 produced six groups
  that reproduce across random seeds (ARI ≥ 0.995), with distinct median profiles on volume, position,
  engagement, consistency, age and update recency."
- **Directional.** "Pages in this group *appear* to share a shape that suggests X." Appear. In this data.
  Under my feature set and my scaling.
- **Decision-support.** "These six groups are a defensible way to route content policy, and here is the
  action I would attach to each and what would make it wrong."
- **A lens, explicitly.** Every cluster is a description of where a page sits in a continuum, not a
  category it belongs to.

### What this work will never be able to say
- **Not semantic clustering.** The release contains **no article text** — no titles, no bodies, no queries.
  I am clustering behavioural metrics and content metadata. Calling this "semantic" or "topic" clustering
  would be describing an analysis I did not do, and the lane guide names it as a common mistake for a
  reason.
- **Not natural kinds.** Six is a choice I made from a silhouette curve that is nearly flat between k=5 and
  k=8. A different k gives different groups. I will show that curve rather than present six as discovered.
- **Not causal, and not predictive.** A cluster does not say what a page *will* do, and an action mapped to
  a cluster is a hypothesis about what a human should look at — never a claim that the action will work.
- **No claim about Google.** Nothing here reveals a ranking factor.
- **Not a mandate to prune.** Cluster membership alone will never justify removing content. Any
  prune-shaped recommendation gets a second, independent check.
- **Not generalisable beyond this panel.** 29 clients, one 90-day window, one content pipeline.

### Leakage discipline — different shape, still required

In [7]:
PRODUCT_FLAGS = ["health_score", "priority_score", "action_type", "refresh_tier",
                 "needs_ctr_fix", "is_quick_win"]
present = [c for c in PRODUCT_FLAGS if c in df.columns]
print(f"Product decision flags in the data: {present if present else 'none - as designed'}")
print("  (if one ever appears, it is background or a baseline to compare against - never a feature,")
print("   because clustering on the product's own answer just rediscovers the product.)\n")

EXCLUDED = {
    "content_id":      "pseudonym - row key only; clustering on an id is meaningless",
    "client_id":       "pseudonym - used to CHECK clusters aren't just clients, never as a feature",
    "trend_pct":       "the shipped pipeline's label source - not my target, and not a shape I want to impose",
    "trend_direction": "same",
    "provider_used":   "generation metadata, not observed search behaviour",
    "model_used":      "generation metadata, not observed search behaviour",
}
for c, why in EXCLUDED.items():
    print(f"  EXCLUDED  {c:18s} {why}")
assert not set(EXCLUDED) & set(FEATURES), "an excluded column got into the feature set"
print("\nGuard passed: no excluded column is in the clustering feature set.")
print(f"Feature set ({len(FEATURES)}): {', '.join(FEATURES)}")

UNSAFE = ["url", "domain", "title", "query", "client_name", "email"]
print(f"\nPublic-safety: columns hinting at raw private text: "
      f"{[c for c in df.columns if any(h in c.lower() for h in UNSAFE)] or 'none'}")
print(f"Id format that will appear in outputs: {df['content_id'].iloc[0]} | {df['client_id'].iloc[0]}")

Product decision flags in the data: none - as designed
  (if one ever appears, it is background or a baseline to compare against - never a feature,
   because clustering on the product's own answer just rediscovers the product.)

  EXCLUDED  content_id         pseudonym - row key only; clustering on an id is meaningless
  EXCLUDED  client_id          pseudonym - used to CHECK clusters aren't just clients, never as a feature
  EXCLUDED  trend_pct          the shipped pipeline's label source - not my target, and not a shape I want to impose
  EXCLUDED  trend_direction    same
  EXCLUDED  provider_used      generation metadata, not observed search behaviour
  EXCLUDED  model_used         generation metadata, not observed search behaviour

Guard passed: no excluded column is in the clustering feature set.
Feature set (9): log_impressions, log_clicks, ctr, avg_position, log_sessions, engagement_rate, impression_consistency, content_age_days, days_since_last_update

Public-safety: columns hi

**The unsupervised-specific trap, named now.** There is no future window here, so I cannot leak the
answer in the usual way. The equivalent failure is subtler: **naming a cluster before inspecting it.** If I
decide in advance that there must be a "hidden gems" group and then find one, I have discovered nothing —
I have projected my expectation onto a partition. Every cluster in this project gets profiled in raw units
first and named second, and the notebook will show the profile above the name so anyone can check I did it
in that order.

**A second one: clusters that are really clients.** With 29 clients and uneven page counts, a "cluster"
could easily just be one big client's house style. Every cluster gets a client-spread check, and any
cluster dominated by a single client is reported as such rather than named as an archetype.

## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere — pseudonymized ids only, verified above
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.

**Provisional, and I mean it.** Lane can be confirmed or changed until the end of Week 4. What would move
me: if the ML-06 signal audit shows the six clusters are mostly separated by *volume* — one thing, measured
six ways — then this is a tier table with extra steps, and Lane 1 (signal analysis) is the honest home for
that finding.

**Next up (ML-03):** turn this into a formal ML task — why clustering rather than classification, what
"success" means when there is no label to be right about, and the unit of analysis as a real dataframe.